# DuckDB Sandbox

Uses the **DuckDB** kernel for native SQL. DuckDB can query Parquet files directly —
no Spark needed for quick ad-hoc analysis on the data lake.

## Query bronze Parquet files directly

In [ ]:
%CREATE :memory:

In [ ]:
SELECT * FROM read_parquet('/workspace/data/lake/bronze/customers/*.parquet')
ORDER BY customer_id

In [ ]:
SELECT * FROM read_parquet('/workspace/data/lake/bronze/orders/*.parquet')
ORDER BY order_id

## Join and aggregate — pure SQL, no Spark overhead

In [ ]:
SELECT
    c.name,
    c.region,
    COUNT(o.order_id) AS num_orders,
    ROUND(SUM(o.quantity * o.unit_price), 2) AS total_spend
FROM read_parquet('/workspace/data/lake/bronze/customers/*.parquet') c
JOIN read_parquet('/workspace/data/lake/bronze/orders/*.parquet') o
    ON c.customer_id = o.customer_id
GROUP BY c.name, c.region
ORDER BY total_spend DESC

## Monthly revenue from bronze orders

In [ ]:
SELECT
    strftime(order_date, '%Y-%m') AS month,
    COUNT(*) AS order_count,
    ROUND(SUM(quantity * unit_price), 2) AS revenue
FROM read_parquet('/workspace/data/lake/bronze/orders/*.parquet')
GROUP BY month
ORDER BY month

## Query gold layer

In [ ]:
SELECT *
FROM read_parquet('/workspace/data/lake/gold/revenue_by_region/*.parquet')
ORDER BY order_month, region

## Create a local DuckDB table from Parquet and query it

In [ ]:
CREATE OR REPLACE TABLE orders AS
SELECT * FROM read_parquet('/workspace/data/lake/bronze/orders/*.parquet');

SELECT product,
       SUM(quantity) AS total_units,
       ROUND(SUM(quantity * unit_price), 2) AS total_revenue
FROM orders
GROUP BY product
ORDER BY total_revenue DESC

## Write query results back to Parquet

In [ ]:
COPY (
    SELECT
        c.region,
        strftime(o.order_date, '%Y-%m') AS month,
        SUM(o.quantity * o.unit_price) AS revenue
    FROM read_parquet('/workspace/data/lake/bronze/customers/*.parquet') c
    JOIN read_parquet('/workspace/data/lake/bronze/orders/*.parquet') o
        ON c.customer_id = o.customer_id
    GROUP BY c.region, month
) TO '/workspace/data/lake/gold/duckdb_revenue_summary.parquet' (FORMAT PARQUET);

SELECT '✅ Written to gold/duckdb_revenue_summary.parquet' AS status